# Comment archiver

This notebook creates an archive of the YouTube comment data retrieved with PYG lean via the API.
The results are stored in a folder specified by the user below.

In [2]:
import os
import zipfile
import json
from collections import defaultdict
from datetime import datetime

# Flags

In [1]:
# Specify actual directory paths
zip_directory = 'D:\\JPMETA'
extract_directory = 'D:\\JPMETA\\extracted'
output_directory = 'D:\\JPMETA\\comments'

# Functions

In [3]:
# Step 1: Extract comment data
def extract_commentsdata(data):
    commentsdata = []
    for key, comments in data.items():  # `data` has multiple keys representing different sets of comments
        for item in comments:
            comment = {
                "id": item.get("id"),
                "snippet": {
                    "textDisplay": item["snippet"].get("textDisplay"),
                    "textOriginal": item["snippet"].get("textOriginal"),
                    "authorDisplayName": item["snippet"].get("authorDisplayName"),
                    "likeCount": item["snippet"].get("likeCount", 0),
                    "publishedAt": item["snippet"].get("publishedAt"),
                    "updatedAt": item["snippet"].get("updatedAt")
                }
            }
            commentsdata.append(comment)
    return commentsdata

# Step 2: Extract comments from ZIP files and add ZIP file source
def extract_comments_from_zips(zip_directory, extract_directory):
    comments_files = defaultdict(list)

    for zip_file in os.listdir(zip_directory):
        if zip_file.endswith('.zip'):
            zip_path = os.path.join(zip_directory, zip_file)
            zip_modification_time = os.path.getmtime(zip_path)

            # Extract files to a unified directory and add ZIP file name as a prefix
            zip_extract_folder = extract_directory
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                for file in zip_ref.namelist():
                    if file.endswith('_comments.json'):
                        # Add ZIP file name as a prefix to the extracted file
                        zip_filename = os.path.splitext(zip_file)[0]  # Get ZIP file name (without extension)
                        extracted_file_path = os.path.join(
                            zip_extract_folder,
                            f"{zip_filename}__{os.path.basename(file)}"
                        )
                        with open(extracted_file_path, 'wb') as f_out:
                            f_out.write(zip_ref.read(file))
                        
                        # Record file path and modification time
                        file_id = os.path.basename(file).split('__')[0]
                        comments_files[file_id].append((extracted_file_path, zip_modification_time))
    return comments_files

# Step 3: Compare and merge comments data
def compare_and_merge_commentsdata(comments_files):
    merged_results = {}

    for file_id, files in comments_files.items():
        # Sort files by modification time
        files.sort(key=lambda x: x[1])

        # Store merged data for each comment ID
        comment_data = defaultdict(list)

        for file_path, modification_time in files:
            timestamp = datetime.fromtimestamp(modification_time).isoformat()

            with open(file_path, 'r', encoding='utf-8') as f:
                current_data = json.load(f)
                current_commentsdata = extract_commentsdata(current_data)

            for comment in current_commentsdata:
                comment_id = comment["id"]

                # If this is the first time encountering the comment ID, add it as the base version
                if not comment_data[comment_id]:
                    comment_data[comment_id].append({
                        "timestamp": timestamp,
                        "snippet": comment["snippet"]
                    })
                else:
                    # Get the last recorded data (last update or initial snippet)
                    last_entry = comment_data[comment_id][-1]

                    # Compare likeCount and updatedAt
                    current_like_count = comment["snippet"]["likeCount"]
                    current_updated_at = comment["snippet"]["updatedAt"]

                    # Get the last recorded likeCount and updatedAt
                    last_like_count = (
                        last_entry.get("likeCount") if "likeCount" in last_entry
                        else last_entry["snippet"]["likeCount"]
                    )
                    last_updated_at = (
                        last_entry.get("updatedAt") if "updatedAt" in last_entry
                        else last_entry["snippet"]["updatedAt"]
                    )

                    # Only add if there's a change
                    if current_like_count != last_like_count or current_updated_at != last_updated_at:
                        comment_data[comment_id].append({
                            "likeCount": current_like_count,
                            "updatedAt": current_updated_at,
                            "timestamp": timestamp
                        })

        # Save results for the current file ID
        merged_results[file_id] = [
            {
                "id": comment_id,
                "updates": updates
            }
            for comment_id, updates in comment_data.items()
        ]

    return merged_results

# Step 4: Save merged comments data
def save_merged_commentsdata(merged_results, output_directory):
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)

    for file_id, data in merged_results.items():
        output_file = os.path.join(output_directory, f'{file_id}_commentsdata.json')
        with open(output_file, 'w', encoding='utf-8') as f_out:
            json.dump(data, f_out, ensure_ascii=True, indent=4)

In [4]:
# Main function
def main(zip_directory, extract_directory, output_directory):
    comments_files = extract_comments_from_zips(zip_directory, extract_directory)
    merged_results = compare_and_merge_commentsdata(comments_files)
    save_merged_commentsdata(merged_results, output_directory)

# Execution section

In [ ]:
main(zip_directory, extract_directory, output_directory)